![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 16 -- Challenge Lab: Animal Image Embeddings + Hierarchical Clustering

This challenge brings together everything you have learned across Days 9--16 into one end-to-end project.

You will:
1. Load a dataset of animal images (10 species)
2. Extract image embeddings using a pretrained ResNet
3. Reduce dimensions with PCA (by explained variance) to remove noise, and visualize with t-SNE
4. Use agglomerative clustering **on the PCA-reduced embeddings** to build a **dendrogram** -- a family tree that reveals which animals the network sees as visually similar

| Concept | Where you learned it |
|---|---|
| Pretrained models, transfer learning | Days 9--10 |
| Image transforms, DataLoaders | Day 9 |
| PCA / t-SNE | Day 14 |
| K-Means, DBSCAN, silhouette score | Day 16 Lab 1 |
| Hierarchical clustering, dendrograms | Day 16 slides |
| Matplotlib visualization | Every day |

---

### The Big Picture

```
Animal Images (10 species)
       |
       v
 +-----------------+
 | Pretrained       |   Extract a 512-dim feature vector
 | ResNet18         |   for each image
 +---------+-------+
           |
           v
 Image Embeddings (512-dim per image)
           |
     +-----+-----+
     |             |
     v             v
 +-----------+  +-----------+
 | PCA       |  | t-SNE     |   2D for visualization
 | (95% var) |  |           |   (scatter plots)
 +-----+-----+  +-----------+
       |
       v
 Reduced Embeddings (n_95 dims)
       |
       v
 +-----------------+
 | Agglomerative    |   Merge closest classes step by step
 | Clustering       |   to build a hierarchy
 +---------+-------+
           |
           v
 Dendrogram ("family tree" of animals)
```

The dendrogram will show which animals the network considers visually similar -- cats near cheetahs, dogs near wolves -- even though nobody told the model about biology.

---
## Setup

In [ ]:
!pip install kagglehub -q

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering

import os
import kagglehub

plt.rcParams['figure.dpi'] = 120

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
## Part 1: Load and Explore the Animal Dataset

We use the **Animals-10** dataset from Kaggle which has ~28,000 images across 10 species:

| Class | Animal |
|---|---|
| 0 | Butterfly |
| 1 | Cat |
| 2 | Chicken |
| 3 | Cow |
| 4 | Dog |
| 5 | Elephant |
| 6 | Horse |
| 7 | Sheep |
| 8 | Spider |
| 9 | Squirrel |

In [ ]:
# Download dataset
data_path = kagglehub.dataset_download('alessiocorrado99/animals10')
print(f"Dataset downloaded to: {data_path}")
print(os.listdir(data_path))

In [ ]:
# The folder names are in Italian -- map them to English
translate = {
    'cane': 'dog', 'cavallo': 'horse', 'elefante': 'elephant',
    'farfalla': 'butterfly', 'gallina': 'chicken', 'gatto': 'cat',
    'mucca': 'cow', 'pecora': 'sheep', 'ragno': 'spider',
    'scoiattolo': 'squirrel'
}

# Find the raw-img directory
img_root = os.path.join(data_path, 'raw-img')
if not os.path.isdir(img_root):
    for d in os.listdir(data_path):
        candidate = os.path.join(data_path, d, 'raw-img')
        if os.path.isdir(candidate):
            img_root = candidate
            break

print(f"Image root: {img_root}")
print(f"Classes found: {sorted(os.listdir(img_root))}")

### Task 1: Create the Dataset and DataLoader

1. Define an ImageNet-style transform pipeline:
   - `Resize(256)` then `CenterCrop(224)`
   - `ToTensor()`
   - `Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])`
2. Load the dataset using `torchvision.datasets.ImageFolder(img_root, transform=...)`
3. The full dataset is large. Create a random subset of **2000 images** using `torch.utils.data.Subset` with random indices
4. Create a `DataLoader` with `batch_size=64` and `shuffle=False`
5. Print the total number of images, number of classes, and the class names

**Tip:** Map the Italian folder names to English using the `translate` dict:
```python
class_names = [translate.get(c, c) for c in dataset.classes]
```

In [ ]:
# Your code here

### Task 2: Visualize Sample Images

Display a grid of **16 random images** (4 rows x 4 columns). For each image:
- Undo the ImageNet normalization so colors look correct: `img = img * std + mean`
- Clip values to [0, 1]
- Show the English class name as the subplot title

**Hint:** The unnormalization formula is:
```python
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img = img.permute(1, 2, 0).numpy() * std + mean
img = np.clip(img, 0, 1)
```

In [ ]:
# Your code here

---
## Part 2: Extract Embeddings with Pretrained ResNet

### Task 3: Build the Feature Extractor

1. Load `resnet18` with pretrained ImageNet weights: `resnet18(weights=ResNet18_Weights.DEFAULT)`
2. Remove the final classification layer (`model.fc`) by replacing it with `nn.Identity()`
3. Set the model to eval mode and move it to `device`
4. Freeze all parameters (set `requires_grad = False`)
5. Verify: pass a dummy batch of shape `(1, 3, 224, 224)` through the model and print the output shape. It should be `(1, 512)`.

```
ResNet18 Architecture (simplified):

Input (3 x 224 x 224)
       |
  Conv layers + ResBlocks
       |
  AdaptiveAvgPool -> (512,)    <-- we keep up to here
       |
  FC layer -> (1000,)          <-- we remove this
```

In [ ]:
# Your code here

### Task 4: Extract All Embeddings

Loop over the DataLoader (with `torch.no_grad()`) and collect:
- `all_embeddings` -- NumPy array of shape `(N, 512)`
- `all_labels` -- NumPy array of shape `(N,)` with integer class labels

Print the shapes and the number of images per class.

```python
all_embeddings = []
all_labels = []

with torch.no_grad():
    for images, labels in dataloader:
        images = images.to(device)
        features = model(images)
        all_embeddings.append(features.cpu().numpy())
        all_labels.append(labels.numpy())

all_embeddings = np.concatenate(all_embeddings)
all_labels = np.concatenate(all_labels)
```

In [ ]:
# Your code here

---
## Part 3: Dimensionality Reduction

### Task 5: PCA -- Find the Right Number of Dimensions

The raw embeddings are 512-dim, but most of that is redundant. Use PCA to find how many dimensions actually matter.

1. Fit PCA with **all components** (`PCA(n_components=512)`) on `all_embeddings`
2. Plot the **cumulative explained variance ratio** as a line plot
3. Add a horizontal dashed line at 95%
4. Find how many components are needed to capture 95% of the variance
5. **Re-fit PCA** with that number of components and transform the embeddings. Store the result as `X_reduced`.
6. Print the shape of `X_reduced`

`X_reduced` is what we will use for clustering in Part 4 -- same information, far fewer dimensions.

In [ ]:
# Your code here

### Task 6: Visualize in 2D -- t-SNE

Apply `TSNE(n_components=2, perplexity=30, random_state=42)` to `all_embeddings` and create a scatter plot:
- Color points by their animal class (`c=all_labels, cmap='tab10'`)
- Add a legend mapping colors to English animal names
- Title: "t-SNE"

Which animals cluster together? Do you see any overlap?

In [ ]:
# Your code here

### Task 7: PCA + t-SNE (Best Practice)

t-SNE is slow on high-dimensional data. The standard trick is to first reduce to ~50 dimensions with PCA, then run t-SNE on the result.

1. Apply `PCA(n_components=50)` to `all_embeddings`
2. Apply `TSNE(n_components=2, perplexity=30, random_state=42)` to the PCA output
3. Plot the result, colored by class with a legend
4. Title: "PCA(50) + t-SNE"

Does this look different from the raw t-SNE in Task 6?

In [ ]:
# Your code here

---
## Part 4: Agglomerative Clustering + Dendrogram

Now we answer the big question: **which animals does ResNet consider most similar?**

We cluster on the **PCA-reduced embeddings** from Task 5 -- not the raw 512-dim vectors. PCA removed the noise dimensions while keeping 95% of the information, which gives the clustering algorithm cleaner signal.

Since a dendrogram with 2000 individual images would be unreadable, we compute one representative vector per class (the mean of all its reduced embeddings), giving us 10 points to cluster.

### Task 8: Build the Dendrogram

1. Compute the **mean reduced embedding** for each class (average all `X_reduced` vectors per animal)
2. Use `scipy.cluster.hierarchy.linkage` with the `ward` method on the 10 class-mean vectors
3. Draw the dendrogram with English animal names as leaf labels

```python
# Step 1: one representative vector per class (using PCA-reduced embeddings)
class_means = np.array([X_reduced[all_labels == i].mean(axis=0) for i in range(len(class_names))])

# Step 2-3: linkage + dendrogram
Z = linkage(class_means, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(Z, labels=class_names, leaf_font_size=12)
plt.title('Animal Similarity Tree (Ward Linkage)')
plt.ylabel('Distance')
plt.show()
```

Which animals are merged first (lowest branch)? Does the tree match biological intuition?

In [ ]:
# Your code here

### Task 9: Compare Linkage Methods

Create a figure with 1 row and 3 columns (`figsize=(20, 6)`) showing dendrograms with three different linkage methods:
- `ward` (minimizes variance when merging)
- `complete` (uses maximum distance between clusters)
- `average` (uses average distance between clusters)

Title each subplot with the linkage method. Do the trees look different? Which produces the most sensible groupings?

In [ ]:
# Your code here

---
## Part 5 -- Bonus: Cut the Tree into Super-Groups

### Task 10: Cut the Dendrogram + Color the Scatter Plot

1. Use `AgglomerativeClustering(n_clusters=4, linkage='ward')` on `class_means` (the PCA-reduced means from Task 8) to get 4 super-groups
2. Print which animals belong to each super-group
3. Map each image back to its super-group (using its class label)
4. Re-plot the 2D t-SNE scatter from Task 6, but color by **super-group** instead of by species. Use a different color per super-group and add a legend showing which animals are in each group.

Do the super-groups make biological sense? (e.g., "four-legged mammals" vs "insects" vs "birds")

In [ ]:
# Your code here

---
## Wrap-Up

**Congratulations!** You built a full pipeline:

- **Data** -- loaded 10 species of animal images
- **Embeddings** -- used a pretrained ResNet to extract 512-dim feature vectors
- **Dimensionality Reduction** -- used PCA (by explained variance) to compress embeddings, then visualized with t-SNE
- **Hierarchical Clustering** -- clustered the PCA-reduced embeddings and built a dendrogram showing which animals the network sees as similar
- **Super-groups** -- cut the tree to discover high-level categories

The key insight: a network trained only to classify ImageNet categories has learned visual features that reflect real biological similarity -- cats look like other small mammals, butterflies look like spiders (small + colorful + detailed patterns), and large four-legged farm animals cluster together.